# Postprocessing pipeline: from MatterGen output to analysis figures

This notebook contains two independent worked workflows:

1. a guided-versus-non-guided Cu–Si–P comparison;
2. a standalone unguided La–Se–C analysis with a manually defined CN(C–C) = 1 target and loss.

Run Sections 0 and 1 once. You can then run either Section 2 or Section 3 independently. The standalone unguided workflow does not use variables, repositories, callables, or results from the guided workflow.

Packaged inputs are under:

```text
vsbtools/materials_dataset/Examples/raw_generations/
```

## 0. Load the installed external environments

The notebook kernel must contain VSBTools. MatterGen supplies coordination descriptors and loss functions; GRACE supplies energy estimates during postprocessing.

When the notebook runs from the VSBTools environment created by `install_vsbtools_mattergen.sh`, MatterGen and GRACE are discovered automatically from the sibling managed environments. The generated `launch_jupyter.sh` also exports their exact paths. No notebook path configuration is required.

For a custom installation layout, set `MATTERGEN_PYTHON_PATH`, `SCOUT_MATTER_SITE_PACKAGES`, and `GRACE_PYTHON` before launching Jupyter.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import warnings
from contextlib import contextmanager
from pathlib import Path

import vsbtools
from vsbtools.materials_dataset.notebook_setup import configure_notebook_external_environments

external_environment = configure_notebook_external_environments(
    quiet_optional_imports=True,
)

VSBTOOLS_PACKAGE = Path(vsbtools.__file__).resolve().parent
print("Python:", sys.executable)
print("VSBTools:", VSBTOOLS_PACKAGE)
for line in external_environment.summary_lines():
    print(line)

## 1. Shared paths and postprocessing utilities

Run the cells in this section once. They define output paths and reusable functions; they do not process any generation.

Each worked workflow below owns its inputs, chemical system, descriptor settings, stages, figures, and manifest.

In [2]:
from importlib.resources import files

EXAMPLES_ROOT = Path(str(files("vsbtools.materials_dataset").joinpath("Examples")))
SCENARIO_FILE = EXAMPLES_ROOT / "scenario_no_relax.yaml"
RUN_ROOT = Path(
    os.environ.get("VSBTOOLS_REPRO_RUN_ROOT", Path.cwd() / "reproducibility_run")
).expanduser().resolve()
PROCESSED_ROOT = RUN_ROOT / "MG_postprocess_pipelines" / "PROCESSED"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

SUPPRESS_PIPELINE_STDERR = True

for required_path in (EXAMPLES_ROOT, SCENARIO_FILE):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print("Examples:", EXAMPLES_ROOT)
print("Processed datasets:", PROCESSED_ROOT)

Examples: /home/vsbat/my_git_projects/vsbtools/vsbtools/materials_dataset/Examples
Processed datasets: /home/vsbat/my_git_projects/vsbtools/vsbtools/materials_dataset/Examples/reproducibility_run/MG_postprocess_pipelines/PROCESSED


### 1.1 Define the reusable postprocessing functions

The first cell adapts a directory to the scenario pipeline. The second accepts either a MatterGen output directory or a single `.zip` archive and returns the exact processed generation repository it created.

In [3]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from vsbtools.materials_dataset.analysis.scenario_pipeline import process_generation_dir
from vsbtools.materials_dataset.io.structures_dataset_io import exploded_zip_tree

BATCH_METADATA_FILE = "input_parameters.txt"


@contextmanager
def suppress_process_stderr(enabled: bool):
    if not enabled:
        yield
        return

    sys.stderr.flush()
    saved_stderr = os.dup(2)
    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(devnull.fileno(), 2)
            yield
    finally:
        os.dup2(saved_stderr, 2)
        os.close(saved_stderr)


def run_pipeline_on_directory(generation_dir: Path) -> Path:
    with suppress_process_stderr(SUPPRESS_PIPELINE_STDERR):
        repo = process_generation_dir(
            generation_dir,
            PROCESSED_ROOT,
            SCENARIO_FILE,
            batch_metadata_file=BATCH_METADATA_FILE,
        )
    if repo is None:
        raise RuntimeError(
            f"No {BATCH_METADATA_FILE!r} file was found under {generation_dir}."
        )
    return repo.root

In [4]:
def process_one_generation(input_path: str | Path) -> Path:
    input_path = Path(input_path).expanduser().resolve()
    if not input_path.exists():
        raise FileNotFoundError(input_path)

    if input_path.is_file() and input_path.suffix.lower() == ".zip":
        with TemporaryDirectory(prefix=f"{input_path.stem}_") as tmp:
            extracted = Path(tmp) / input_path.stem
            extracted.mkdir(parents=True, exist_ok=True)
            with ZipFile(input_path) as archive:
                archive.extractall(extracted)
            return run_pipeline_on_directory(extracted)

    if input_path.is_dir():
        with exploded_zip_tree(input_path) as extracted:
            source = (
                extracted
                if any(extracted.rglob(BATCH_METADATA_FILE))
                else input_path
            )
            return run_pipeline_on_directory(source)

    raise ValueError(f"Expected a generation directory or .zip archive: {input_path}")

### 1.2 Automatic metadata inference and the manual fallback

For a guided generation, `callables_from_ds(parse_raw_dataset)` reads `batch_metadata["guidance"]` and reconstructs:

- the scalar descriptor callable or callables;
- the matching target value for each descriptor;
- the MatterGen loss callable and its zero-loss target;
- the guidance name, including supported legacy aliases.

That automatic path is used throughout Section 2. The user only chooses which inferred descriptor column to display when metadata defines more than one.

For an unguided generation, `batch_metadata["guidance"]` is `None`, so there is nothing to infer. Section 3 demonstrates the manual fallback with `get_target_value_fn(name, **params)` and `get_loss_fn(name, **params)`.

The current VSBTools MatterGen bridge exposes these public scalar descriptors for manual construction:

| Descriptor name | Typical parameters |
|---|---|
| `volume` | `{}` |
| `volume_pa` | `{}` |
| `compute_mean_coordination` | `type_A`, `type_B`; optional `r_cut`, `alpha` |
| `compute_target_coordination_share` | `type_A`, `type_B`, `target`; optional `tau`, `r_cut`, `alpha` |
| `compute_target_share` | Backward-compatible alias of `compute_target_coordination_share` |

For coordination descriptors, `type_A` and `type_B` accept atomic numbers or collections of atomic numbers, which also supports grouped species.

Losses are discovered from the installed MatterGen `LOSS_REGISTRY`. The supplied MatterGen fork commonly provides `volume`, `volume_pa`, `mean_coordination`, `target_coordination_share`, `target_coordination`, `ranked_coordination`, `group_coordination`, `group_target_coordination`, `environment`, and `dominant_environment`. Their accepted parameter dictionaries are defined by the configured MatterGen checkout.

## 2. Guided-versus-non-guided example

This workflow processes the packaged Cu–Si–P inputs and compares a guided target with its non-guided baseline.

The descriptor, target, and loss are reconstructed from the guided run's `batch_metadata["guidance"]`. The `plot` block contains presentation choices only: it selects one inferred descriptor column and controls its margin, label, and histogram range. Set `plot.column` to `None` when exactly one descriptor is inferred, or name a column when the metadata defines several.

In [12]:
# USER INPUTS for the guided-versus-non-guided workflow.
GUIDED_CASE = {
    "inputs": [
        EXAMPLES_ROOT / "raw_generations" / "Cu-Si-P" / "Cu-Si-P_nonguided.zip",
        EXAMPLES_ROOT / "raw_generations" / "Cu-Si-P" / "Cu-Si-P_guided_CuP3.zip",
    ],
    "system": "Cu-Si-P",
    # Presentation choices for one descriptor inferred from batch metadata.
    "plot": {
        "column": None,  # Select a name only when metadata defines several descriptors.
        "margin": 0.2,
        "axis_label": None,  # None uses the inferred column name.
        "max_bincenter": 10,
    },
    "hist_stage": "symmetrize_raw",
    "pvalue_stage": "symmetrize_raw",
    "reference_stage": "poll_db",
    "pareto_stages": ["add_ref_deduplicated"],
    "n_pareto_fronts": 3,
    "trim_loss": None,
    "trim_ehull": 0.3,
}

guided_output_dir = RUN_ROOT / "analysis_outputs" / GUIDED_CASE["system"]
guided_output_dir.mkdir(parents=True, exist_ok=True)

for input_path in GUIDED_CASE["inputs"]:
    if not Path(input_path).exists():
        raise FileNotFoundError(input_path)

### 2.1 Process the Cu–Si–P inputs

In [ ]:
guided_repos = [
    process_one_generation(input_path)
    for input_path in GUIDED_CASE["inputs"]
]
guided_system_dirs = {repo.parent for repo in guided_repos}
if len(guided_system_dirs) != 1:
    raise RuntimeError(f"Inputs produced repositories in different systems: {guided_system_dirs}")

guided_system_dir = guided_system_dirs.pop()
print("Processed generation repositories:")
for repo in guided_repos:
    print(" -", repo)

### 2.2 Infer callables from the guided batch metadata

The processed `parse_raw` dataset retains the original batch metadata. `callables_from_ds` turns its guidance description into ready-to-use descriptor and loss callables plus their targets.

In [ ]:
from vsbtools.materials_dataset.analysis.guidance_statistics import (
    callables_from_ds,
)
from vsbtools.materials_dataset.scripts.build_tables import (
    stage_datasets_from_repo,
)

guided_parse_ds = None
for repo in guided_repos:
    candidate = stage_datasets_from_repo(repo)["parse_raw"]
    guidance = candidate.metadata["batch_metadata"].get("guidance")
    if isinstance(guidance, dict) and guidance:
        guided_parse_ds = candidate
        break

if guided_parse_ds is None:
    raise RuntimeError("No guided repository was found among GUIDED_CASE['inputs'].")

guided_callables, guided_targets, guided_guidance_name = callables_from_ds(
    guided_parse_ds
)
guided_descriptor_callables = {
    name: fn
    for name, fn in guided_callables.items()
    if not name.startswith("loss_")
}
guided_loss_callables = {
    name: fn
    for name, fn in guided_callables.items()
    if name.startswith("loss_")
}

plot_column = GUIDED_CASE["plot"]["column"]
if plot_column is None:
    if len(guided_descriptor_callables) != 1:
        raise RuntimeError(
            "Metadata defines multiple descriptors; set GUIDED_CASE['plot']['column'] "
            f"to one of {list(guided_descriptor_callables)}."
        )
    plot_column = next(iter(guided_descriptor_callables))
elif plot_column not in guided_descriptor_callables:
    raise KeyError(
        f"Unknown plot column {plot_column!r}; inferred columns are "
        f"{list(guided_descriptor_callables)}."
    )

print("Guidance:", guided_guidance_name)
print("Descriptor columns:", list(guided_descriptor_callables))
print("Loss columns:", list(guided_loss_callables))
print("Targets:", guided_targets)
print("Selected plot column:", plot_column)

### 2.3 Build summary and Pareto artifacts

The summary builder receives the callables reconstructed in Section 2.2 and reuses them for both the guided and matched non-guided repositories, keeping their columns comparable.

In [ ]:
from vsbtools.materials_dataset.scripts.build_tables import (
    build_guidance_summary_for_processed_system,
)

guided_summary_report = build_guidance_summary_for_processed_system(
    guided_system_dir,
    target_stages=GUIDED_CASE["pareto_stages"],
    auto_ref_stages=True,
    callables=guided_callables,
    max_pareto_front=GUIDED_CASE["n_pareto_fronts"],
    return_report=True,
)
print(guided_summary_report)

### 2.4 Plot a metadata-derived descriptor

The histogram consumes the inferred callable directly. Only the displayed column and plotting options come from `GUIDED_CASE["plot"]`.

In [ ]:
from vsbtools.materials_dataset.analysis.guidance_statistics import (
    collect_stage_dataset_dict,
    histo_data_collection,
)

guided_plot = GUIDED_CASE["plot"]
guided_hist_datasets = collect_stage_dataset_dict(
    guided_repos,
    stage=GUIDED_CASE["hist_stage"],
    ref_stage=GUIDED_CASE["reference_stage"],
)
guided_histograms = histo_data_collection(
    guided_hist_datasets,
    fn=guided_descriptor_callables[plot_column],
    filter_max_el=False,
    max_bincenter=guided_plot["max_bincenter"],
)

print("Descriptor column:", plot_column)
print("Target:", guided_targets[plot_column])
print("Datasets:", list(guided_hist_datasets))

In [ ]:
import matplotlib.pyplot as plt

from vsbtools.materials_dataset.analysis.guidance_statistics import (
    plot_multihistogram,
)

guided_hist_figure, guided_hist_ax = plot_multihistogram(
    multidata=guided_histograms,
    target=guided_targets[plot_column],
    max_bincenter=guided_plot["max_bincenter"],
    show_gaussian=True,
    simplified_legend=True,
)
guided_hist_ax.set_xlabel(guided_plot["axis_label"] or plot_column)
guided_hist_ax.set_box_aspect(1)

guided_hist_path = guided_output_dir / (
    f"{GUIDED_CASE['system']}_{plot_column}_"
    f"{GUIDED_CASE['hist_stage']}_histogram.pdf"
)
guided_hist_figure.savefig(
    guided_hist_path, bbox_inches="tight", pad_inches=0.1
)
print("Saved:", guided_hist_path)

### 2.5 Test enrichment of the selected descriptor

The same descriptor callable and target inferred from metadata define a matching structure. `plot.margin` supplies the analysis tolerance. The full configured raw/symmetrized stage is the denominator, and the one-sided pooled two-proportion z-test uses `guided > non-guided`.

In [10]:
from vsbtools.materials_dataset.analysis.guidance_statistics import (
    collect_stage_dataset_dict,
    get_two_proportion_z_test,
)


def is_generated_guided(label: str) -> bool:
    return label != "Non-guided" and label.lower() != "reference"


def count_matching_entries(ds, callables, targets, margins) -> int:
    return sum(
        all(
            abs(fn(entry) - targets[name]) <= margins[name]
            for name, fn in callables.items()
        )
        for entry in ds
    )

In [ ]:
target_callables = {
    plot_column: guided_descriptor_callables[plot_column]
}
targets = {
    plot_column: guided_targets[plot_column]
}
margins = {
    plot_column: GUIDED_CASE["plot"]["margin"]
}

print("Descriptor column:", plot_column)
print("Targets:", targets)
print("Margins:", margins)

In [ ]:
pvalue_datasets = collect_stage_dataset_dict(
    guided_repos,
    stage=GUIDED_CASE["pvalue_stage"],
    ref_stage=GUIDED_CASE["reference_stage"],
)
if "Non-guided" not in pvalue_datasets:
    raise RuntimeError("The matched repositories do not contain a non-guided dataset.")

non_guided_ds = pvalue_datasets["Non-guided"]
non_guided_hits = count_matching_entries(
    non_guided_ds, target_callables, targets, margins
)
pvalue_rows = []

for label, guided_ds in pvalue_datasets.items():
    if not is_generated_guided(label):
        continue

    stats = get_two_proportion_z_test(
        non_guided_ds,
        target_callables,
        targets,
        margins,
        ds_guided=guided_ds,
        alternative="greater",
    )
    guided_hits = count_matching_entries(
        guided_ds, target_callables, targets, margins
    )
    pvalue_rows.append({
        "guided_label": label,
        "nonguided_hits": non_guided_hits,
        "nonguided_total": len(non_guided_ds),
        "nonguided_fraction": non_guided_hits / len(non_guided_ds),
        "guided_hits": guided_hits,
        "guided_total": len(guided_ds),
        "guided_fraction": guided_hits / len(guided_ds),
        **stats,
    })

In [ ]:
import pandas as pd
from IPython.display import display

pvalue_df = pd.DataFrame(pvalue_rows)
pvalue_path = guided_output_dir / (
    f"{GUIDED_CASE['system']}_{plot_column}_"
    f"{GUIDED_CASE['pvalue_stage']}_pvalues.csv"
)
pvalue_df.to_csv(pvalue_path, index=False)
display(pvalue_df)
print("Saved:", pvalue_path)

### 2.6 Plot the guided Pareto fronts

This cell reads the Pareto CSV files created by the guided-comparison summary cell and saves one figure per loss/stage combination.

In [ ]:
guided_pareto_jobs = []
for stage_name in GUIDED_CASE["pareto_stages"]:
    for stage_dir in guided_system_dir.rglob(f"{stage_name}_*"):
        for first_front in sorted(stage_dir.glob("*pf_1.csv")):
            prefix = first_front.name.replace("pf_1.csv", "")
            loss_suffix = prefix.strip("_")
            loss_column = f"loss_{loss_suffix}" if loss_suffix else "loss"
            guided_pareto_jobs.append(
                (stage_dir, prefix, loss_column)
            )

print("Pareto plots to build:", len(guided_pareto_jobs))

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from vsbtools.materials_dataset.analysis.pareto_fronts import plot_pareto

guided_pareto_paths = []
for stage_dir, prefix, loss_column in guided_pareto_jobs:
    try:
        pareto_ax = plot_pareto(
            stage_dir,
            col1=loss_column,
            col2="e_hull/at",
            trim_col1=GUIDED_CASE["trim_loss"],
            trim_col2=GUIDED_CASE["trim_ehull"],
            n_fronts=GUIDED_CASE["n_pareto_fronts"],
            prefix=prefix,
            article_axes=True,
            show_title=False,
        )
    except Exception as exc:
        warnings.warn(f"Skipping {stage_dir}: {exc}")
        continue

    figure_path = guided_output_dir / (
        f"{stage_dir.parent.name}_{stage_dir.name}_"
        f"{prefix}{GUIDED_CASE['n_pareto_fronts']}_pareto_fronts.pdf"
    )
    pareto_ax.figure.savefig(
        figure_path, bbox_inches="tight", pad_inches=0.1
    )
    display(pareto_ax.figure)
    plt.close(pareto_ax.figure)
    guided_pareto_paths.append(figure_path)

print("Pareto figures:")
for figure_path in guided_pareto_paths:
    print(" -", figure_path)

### 2.7 Save the guided-workflow manifest

In [ ]:
guided_manifest = {
    "python": sys.executable,
    "vsbtools_package": str(VSBTOOLS_PACKAGE),
    "mattergen_python_path": os.environ.get("MATTERGEN_PYTHON_PATH"),
    "grace_python": os.environ.get("GRACE_PYTHON"),
    "scenario_file": str(SCENARIO_FILE),
    "processed_root": str(PROCESSED_ROOT),
    "case": {
        **GUIDED_CASE,
        "inputs": [str(path) for path in GUIDED_CASE["inputs"]],
    },
    "inferred_guidance": guided_guidance_name,
    "inferred_descriptor_columns": list(guided_descriptor_callables),
    "inferred_loss_columns": list(guided_loss_callables),
    "inferred_targets": guided_targets,
    "processed_repositories": [str(repo) for repo in guided_repos],
    "histogram": str(guided_hist_path),
    "pvalues": str(pvalue_path),
    "pareto_figures": [str(path) for path in guided_pareto_paths],
}
guided_manifest_path = (
    guided_output_dir
    / f"{GUIDED_CASE['system']}_guided_comparison_manifest.json"
)
guided_manifest_path.write_text(
    json.dumps(guided_manifest, indent=2),
    encoding="utf-8",
)
print("Saved:", guided_manifest_path)

## 3. Standalone unguided-generation analysis

This workflow uses only the shared setup and utilities from Sections 0 and 1.

Here `batch_metadata["guidance"]` is `None`, so the descriptor and loss must be supplied manually. The `descriptor` and `loss` blocks deliberately select `compute_mean_coordination` and `environment` for the CN(C–C) = 1 example. To use another guidance kind, select names from Section 1.2 and replace their complete parameter dictionaries, output-column names, target, and plot settings.

In [6]:
# USER INPUTS for the standalone unguided workflow.
UNGUIDED_CASE = {
    "input": (
        EXAMPLES_ROOT
        / "raw_generations"
        / "La-Se-C_non_guided_only"
        / "La-Se-C_nonguided.zip"
    ),
    "system": "La-Se-C",
    # Deliberate choice: calculate and plot the mean C-C coordination.
    "descriptor": {
        "name": "compute_mean_coordination",
        "params": {"type_A": 6, "type_B": 6},
        "column": "C-C",
        "target": 1,
        "axis_label": "mean CN(C-C)",
        "max_bincenter": 10,
    },
    # Deliberate choice: evaluate the matching environment loss.
    "loss": {
        "name": "environment",
        "params": {"target": {"mode": "huber", "C-C": 1}},
        "column": "loss_environment_mode_huber__C-C_1",
    },
    "hist_stage": "symmetrize_raw",
    "reference_stage": "poll_db",
    "pareto_stage": "add_ref_deduplicated",
    "n_pareto_fronts": 3,
    "trim_loss": None,
    "trim_ehull": 0.3,
}

### 3.1 Process and inspect the unguided archive

In [ ]:
from vsbtools.materials_dataset.scripts.build_tables import stage_datasets_from_repo

unguided_input = Path(UNGUIDED_CASE["input"]).expanduser().resolve()
if not unguided_input.is_file():
    raise FileNotFoundError(unguided_input)

unguided_repo = process_one_generation(unguided_input)
unguided_datasets = stage_datasets_from_repo(unguided_repo)
unguided_metadata = unguided_datasets["parse_raw"].metadata["batch_metadata"]

metadata_guidance = unguided_metadata.get("guidance")
if isinstance(metadata_guidance, str) and metadata_guidance.strip().lower() in {"", "none", "null"}:
    metadata_guidance = None
if metadata_guidance is not None:
    raise RuntimeError(
        "Expected an unguided generation; found "
        f"guidance metadata {metadata_guidance!r}"
    )

unguided_output_dir = RUN_ROOT / "analysis_outputs" / UNGUIDED_CASE["system"]
unguided_output_dir.mkdir(parents=True, exist_ok=True)
print("Processed repository:", unguided_repo)
print("Batch metadata guidance:", repr(metadata_guidance))

### 3.2 Inspect and instantiate the selected descriptor and loss

These cells read the callable names and parameter dictionaries exactly as written in `UNGUIDED_CASE`, then pass them to `get_target_value_fn` and `get_loss_fn`.

In [7]:
descriptor_spec = UNGUIDED_CASE["descriptor"]
descriptor_name = str(descriptor_spec["name"])
descriptor_params = dict(descriptor_spec["params"])
descriptor_column = str(descriptor_spec["column"])
descriptor_target = descriptor_spec["target"]

print("Descriptor:", descriptor_name)
print("Parameters:", descriptor_params)
print("Output column:", descriptor_column)
print("Target:", descriptor_target)

Descriptor: compute_mean_coordination
Parameters: {'type_A': 6, 'type_B': 6}
Output column: C-C
Target: 1


In [8]:
loss_spec = UNGUIDED_CASE["loss"]
loss_name = str(loss_spec["name"])
loss_params = dict(loss_spec["params"])
loss_column = str(loss_spec["column"])

print("Loss:", loss_name)
print("Parameters:", loss_params)
print("Output column:", loss_column)

Loss: environment
Parameters: {'target': {'mode': 'huber', 'C-C': 1}}
Output column: loss_environment_mode_huber__C-C_1


In [9]:
from vsbtools.materials_dataset.analysis.guidance_statistics import (
    get_loss_fn,
    get_target_value_fn,
)

unguided_callables = {
    descriptor_column: get_target_value_fn(
        descriptor_name,
        force_gpu=0,
        **descriptor_params,
    ),
    loss_column: get_loss_fn(
        loss_name,
        force_gpu=0,
        **loss_params,
    ),
}

### 3.3 Build summary and Pareto artifacts for this repository

In [ ]:
from vsbtools.materials_dataset.scripts.build_tables import (
    build_guidance_summary_for_repo,
)

unguided_summary_callables = build_guidance_summary_for_repo(
    unguided_repo,
    target_stages=[UNGUIDED_CASE["pareto_stage"]],
    auto_ref_stages=True,
    callables=unguided_callables,
    max_pareto_front=UNGUIDED_CASE["n_pareto_fronts"],
)
print("Summary columns:", list(unguided_summary_callables))

### 3.4 Calculate and plot the descriptor distribution

In [ ]:
from vsbtools.materials_dataset.analysis.guidance_statistics import (
    collect_stage_dataset_dict,
    histo_data_collection,
)

unguided_hist_datasets = collect_stage_dataset_dict(
    [unguided_repo],
    stage=UNGUIDED_CASE["hist_stage"],
    ref_stage=UNGUIDED_CASE["reference_stage"],
)
unguided_histograms = histo_data_collection(
    unguided_hist_datasets,
    callable_name=descriptor_name,
    callable_params=descriptor_params,
    filter_max_el=False,
    max_bincenter=descriptor_spec["max_bincenter"],
)
if not any(item["counts"] is not None for item in unguided_histograms):
    raise RuntimeError(
        f"No values were calculated for {descriptor_column!r}."
    )

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from vsbtools.materials_dataset.analysis.guidance_statistics import (
    plot_multihistogram,
)

unguided_hist_figure, unguided_hist_ax = plot_multihistogram(
    multidata=unguided_histograms,
    target=descriptor_target,
    max_bincenter=descriptor_spec["max_bincenter"],
    show_gaussian=True,
    simplified_legend=True,
    legend_kwargs={
        "labelspacing": 0.2,
        "loc": "upper center",
        "bbox_to_anchor": (0.5, -0.12),
    },
)
unguided_hist_ax.set_xlabel(descriptor_spec["axis_label"])
unguided_hist_ax.set_box_aspect(1)

unguided_hist_path = unguided_output_dir / (
    f"{UNGUIDED_CASE['system']}_nonguided_"
    f"{descriptor_column}_distribution.pdf"
)
unguided_hist_figure.savefig(
    unguided_hist_path, bbox_inches="tight", pad_inches=0.1
)
display(unguided_hist_figure)
plt.close(unguided_hist_figure)
print("Saved:", unguided_hist_path)

### 3.5 Plot the manual-loss Pareto fronts

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from vsbtools.materials_dataset.analysis.pareto_fronts import plot_pareto

unguided_stage_dir = Path(
    unguided_datasets[UNGUIDED_CASE["pareto_stage"]].base_path
)
unguided_pareto_ax = plot_pareto(
    unguided_stage_dir,
    col1=loss_column,
    col2="e_hull/at",
    trim_col1=UNGUIDED_CASE["trim_loss"],
    trim_col2=UNGUIDED_CASE["trim_ehull"],
    n_fronts=UNGUIDED_CASE["n_pareto_fronts"],
    prefix="",
    article_axes=True,
    show_title=False,
)
unguided_pareto_path = unguided_output_dir / (
    f"{UNGUIDED_CASE['system']}_nonguided_"
    f"{descriptor_column}_{descriptor_target}_pareto_fronts.pdf"
)
unguided_pareto_ax.figure.savefig(
    unguided_pareto_path, bbox_inches="tight", pad_inches=0.1
)
display(unguided_pareto_ax.figure)
plt.close(unguided_pareto_ax.figure)
print("Saved:", unguided_pareto_path)

### 3.6 Save the standalone unguided-workflow manifest

In [ ]:
unguided_manifest = {
    "python": sys.executable,
    "vsbtools_package": str(VSBTOOLS_PACKAGE),
    "mattergen_python_path": os.environ.get("MATTERGEN_PYTHON_PATH"),
    "grace_python": os.environ.get("GRACE_PYTHON"),
    "scenario_file": str(SCENARIO_FILE),
    "processed_root": str(PROCESSED_ROOT),
    "case": {
        **UNGUIDED_CASE,
        "input": str(UNGUIDED_CASE["input"]),
    },
    "processed_repository": str(unguided_repo),
    "batch_metadata_guidance": metadata_guidance,
    "descriptor": {
        "name": descriptor_name,
        "params": descriptor_params,
        "target": descriptor_target,
    },
    "loss": {
        "name": loss_name,
        "params": loss_params,
        "column": loss_column,
    },
    "histogram": str(unguided_hist_path),
    "pareto_figure": str(unguided_pareto_path),
}
unguided_manifest_path = unguided_output_dir / (
    f"{UNGUIDED_CASE['system']}_unguided_analysis_manifest.json"
)
unguided_manifest_path.write_text(
    json.dumps(unguided_manifest, indent=2),
    encoding="utf-8",
)
print("Saved:", unguided_manifest_path)